# MedGemma-4B
- Environment: envs/medical.yaml
- Runtime: paid Colab / 24GB GPU
- GPU/VRAM: 16-24GB
- Quantization: 4-bit on T4
- Known issues: medical stack conflicts with generic env

In [ ]:
import subprocess
import sys
from pathlib import Path

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/content/LLMComparison'),
    Path('/content/drive/MyDrive/LLMComparison'),
]

PROJECT_ROOT = next(
    (
        root
        for root in candidate_roots
        if (root / 'experiments').exists() and (root / 'src').exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Clone into /content/LLMComparison or mount Drive first.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')


In [ ]:
MODEL = 'medgemma-4b'
PRESET = 'colab_paid_mid'
DATASETS = ['hf_iu_xray', 'hf_vqa_rad']
NUM_SAMPLES = 20
OUTPUT_DIR = PROJECT_ROOT / 'results'
RUN_NAME = 'medgemma_4b_multi'

print('Model:', MODEL)
print('Preset:', PRESET)
print('Run name:', RUN_NAME)
print('Output dir:', OUTPUT_DIR)


In [ ]:
import re
import time

command = [
    sys.executable,
    str(PROJECT_ROOT / 'experiments' / 'run_unified.py'),
    '--preset', PRESET,
    '--models', MODEL,
    '--datasets', *DATASETS,
    '--num-samples', str(NUM_SAMPLES),
    '--skip-inaccessible',
    '--output-dir', str(OUTPUT_DIR),
    '--run-name', RUN_NAME,
]

print('Running command:')
print(command)

completed_samples = 0
total_samples = None
start_ts = time.time()

process = subprocess.Popen(
    command,
    cwd=PROJECT_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    text_line = line.rstrip()

    sample_match = re.search(r'Progress model=.* sample=(\d+)/(\d+)', text_line)
    if sample_match:
        completed_samples = int(sample_match.group(1))
        total_samples = int(sample_match.group(2))

    elapsed = time.time() - start_ts
    elapsed_min = elapsed / 60.0

    if total_samples and completed_samples > 0:
        avg_per_sample = elapsed / completed_samples
        remaining_samples = max(0, total_samples - completed_samples)
        eta_sec = avg_per_sample * remaining_samples
        eta_min = eta_sec / 60.0
        print(
            f'[sample {completed_samples}/{total_samples}] elapsed={elapsed_min:.1f}m eta~{eta_min:.1f}m | {text_line}'
        )
    else:
        print(f'[sample ?/?] elapsed={elapsed_min:.1f}m eta~unknown | {text_line}')

return_code = process.wait()
total_elapsed_min = (time.time() - start_ts) / 60.0
print(f'Finished with code={return_code} in {total_elapsed_min:.1f}m')

if return_code != 0:
    raise RuntimeError(f'run_unified failed with exit code {return_code}')
